# 2. Предобработка данных (Preprocessing)

Входные данные: `data/churn_cleaned.csv` — очищенный датасет из этапа EDA.

Структура раздела:
- 2.1 Загрузка данных
- 2.2 Кодирование категориальных признаков (One-Hot Encoding)
- 2.3 Разделение на train/test
- 2.4 Масштабирование числовых признаков
- 2.5 Сохранение финального датасета

## 2.1 Загрузка данных

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/churn_cleaned.csv')

print(f"Размер датасета: {df.shape}")
print(f"\nТипы данных:\n{df.dtypes}")
print(f"\nПервые строки:\n{df.head(3)}")

Размер датасета: (15000, 12)

Типы данных:
кредитный_рейтинг     float64
город                     str
пол                       str
возраст               float64
стаж_в_банке          float64
баланс_депозита       float64
число_продуктов       float64
есть_кредитка         float64
активный_клиент       float64
оценочная_зарплата    float64
ушел_из_банка         float64
есть_депозит            int64
dtype: object

Первые строки:
   кредитный_рейтинг   город     пол  возраст  стаж_в_банке  баланс_депозита  \
0              754.0  Астана    Male     40.0           8.0        102954.68   
1              579.0  Алматы  Female     28.0           1.0             0.00   
2              744.0  Алматы  Female     56.0           5.0             0.00   

   число_продуктов  есть_кредитка  активный_клиент  оценочная_зарплата  \
0              2.0            1.0              1.0           149238.35   
1              2.0            1.0              0.0            64869.32   
2              1.0      

## 2.2 Кодирование категориальных признаков

Признаки `город` и `пол` — номинальные категории без порядка,
применяем One-Hot Encoding. Используем `drop='first'` для избежания
мультиколлинеарности (dummy variable trap).

In [8]:
df_encoded = pd.get_dummies(df, columns=['город', 'пол'], drop_first=True)

# Приводим столбцы булиан к int
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print("Новые признаки после OHE:")
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print(new_cols)
print(f"\nИтоговое число признаков: {df_encoded.shape[1]}")
print(df_encoded.head(3))

Новые признаки после OHE:
['город_Астана', 'город_Атырау', 'пол_Male']

Итоговое число признаков: 13
   кредитный_рейтинг  возраст  стаж_в_банке  баланс_депозита  число_продуктов  \
0              754.0     40.0           8.0        102954.68              2.0   
1              579.0     28.0           1.0             0.00              2.0   
2              744.0     56.0           5.0             0.00              1.0   

   есть_кредитка  активный_клиент  оценочная_зарплата  ушел_из_банка  \
0            1.0              1.0           149238.35            0.0   
1            1.0              0.0            64869.32            0.0   
2            1.0              0.0           158816.03            1.0   

   есть_депозит  город_Астана  город_Атырау  пол_Male  
0             1             1             0         1  
1             0             0             0         0  
2             0             0             0         0  


## 2.3 Разделение на train/test

Делим в соотношении 80/20 со стратификацией по целевой переменной —
это гарантирует одинаковую долю оттока (~20.4%) в обеих выборках.
`random_state=42` фиксирует воспроизводимость результатов.

In [9]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['ушел_из_банка'])
y = df_encoded['ушел_из_банка']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} строк ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} строк ({X_test.shape[0]/len(df)*100:.1f}%)")
print(f"\nДоля оттока в train: {y_train.mean():.3f}")
print(f"Доля оттока в test:  {y_test.mean():.3f}")

Train: 12000 строк (80.0%)
Test:  3000 строк (20.0%)

Доля оттока в train: 0.204
Доля оттока в test:  0.204


## 2.4 Масштабирование числовых признаков

Масштабирование необходимо для линейных моделей (Logistic Regression),
которую будем использовать как baseline. Tree-based модели (CatBoost, XGBoost)
масштабирования не требуют, но единый пайплайн упрощает код.

Применяем `StandardScaler` — только на train, затем трансформируем test.
Это исключает data leakage: тестовая выборка не влияет на параметры скейлера.

In [10]:
from sklearn.preprocessing import StandardScaler
import joblib

num_cols = ['кредитный_рейтинг', 'возраст', 'стаж_в_банке',
            'баланс_депозита', 'оценочная_зарплата']

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

joblib.dump(scaler, '../models/scaler.pkl') # Сохраняем скейлер — он понадобится в API для инференса

print("Скейлер сохранён в models/scaler.pkl")
print(f"\nПараметры скейлера (mean):\n{dict(zip(num_cols, scaler.mean_.round(2)))}")

Скейлер сохранён в models/scaler.pkl

Параметры скейлера (mean):
{'кредитный_рейтинг': np.float64(659.08), 'возраст': np.float64(37.89), 'стаж_в_банке': np.float64(5.02), 'баланс_депозита': np.float64(43236.58), 'оценочная_зарплата': np.float64(118372.66)}


## 2.5 Сохранение финального датасета

In [11]:
X_train_scaled.to_csv('../data/train_test/X_train.csv', index=False)
X_test_scaled.to_csv('../data/train_test/X_test.csv', index=False)
y_train.to_csv('../data/train_test/y_train.csv', index=False)
y_test.to_csv('../data/train_test/y_test.csv', index=False)


# Нескейлированные версии для tree-based моделей
X_train.to_csv('../data/train_test/X_train_raw.csv', index=False)
X_test.to_csv('../data/train_test/X_test_raw.csv', index=False)

print("Все файлы сохранены в папку data/")
print(f"\nИтоговые признаки для модели ({X_train.shape[1]} шт.):")
print(list(X_train.columns))

Все файлы сохранены в папку data/

Итоговые признаки для модели (12 шт.):
['кредитный_рейтинг', 'возраст', 'стаж_в_банке', 'баланс_депозита', 'число_продуктов', 'есть_кредитка', 'активный_клиент', 'оценочная_зарплата', 'есть_депозит', 'город_Астана', 'город_Атырау', 'пол_Male']


## Итоги предобработки

- Категориальные признаки `город` и `пол` закодированы через OHE
- Выборка разделена 80/20 со стратификацией — доля оттока сохранена в обеих частях
- Числовые признаки масштабированы `StandardScaler` (fit только на train)
- Сохранены две версии: scaled (для Logistic Regression) и raw (для CatBoost/XGBoost)
- Скейлер сохранён для последующего использования в API

**Следующий шаг:** обучение и сравнение моделей (`03_modeling.ipynb`)